In [51]:
import pandas as pd

# Colonnes du dataset  récupéré depuis le readme.md du dataset
COLONNES = [
    "id",                   
    "label",                
    "statement",            
    "subject",              
    "speaker",              
    "job_title",            
    "state",                
    "party",                
    "barely_true_counts",   
    "false_counts",         
    "half_true_counts",     
    "mostly_true_counts",   
    "pants_on_fire_counts",
    "context"               
]

# Chargement des données 
train = pd.read_csv("data/train.tsv", sep="\t", header=None, names=COLONNES)
valid = pd.read_csv("data/valid.tsv", sep="\t", header=None, names=COLONNES)
test  = pd.read_csv("data/test.tsv",  sep="\t", header=None, names=COLONNES)

# Vérifiation
print("Chargement réussi !")
print(f"   Train : {train.shape[0]} lignes, {train.shape[1]} colonnes")
print(f"   Valid : {valid.shape[0]} lignes, {valid.shape[1]} colonnes")
print(f"   Test  : {test.shape[0]} lignes, {test.shape[1]} colonnes")

Chargement réussi !
   Train : 10240 lignes, 14 colonnes
   Valid : 1284 lignes, 14 colonnes
   Test  : 1267 lignes, 14 colonnes


VÉRIFICATION DES COLONNES

In [52]:
print("Colonnes Train :", train.columns.tolist())
print("Colonnes Valid :", valid.columns.tolist())
print("Colonnes Test :", test.columns.tolist())

# Vérification automatiqu
if train.columns.tolist() == valid.columns.tolist() == test.columns.tolist():
    print("\n Les 3 datasets ont exactement les mêmes colonnes !")
else:
    print("\n Attention, les colonnes sont différentes !")

Colonnes Train : ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']
Colonnes Valid : ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']
Colonnes Test : ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context']

 Les 3 datasets ont exactement les mêmes colonnes !


Apercu des données

In [53]:
print("Apercu des 5 premières lignes du dataset train :")
train.head(5)

Apercu des 5 premières lignes du dataset train :


,id,label,statement,subject,speaker,job_title,state,party,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN


 VALEURS MANQUANTES SUR LES 3 DATASETS




In [54]:
import pandas as pd
from IPython.display import display

for nom, df in [("Train", train), ("Valid", valid), ("Test", test)]:
    print(f"\n### {nom}")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    result = pd.DataFrame({
        "Valeurs manquantes": missing,
        "Pourcentage (%)": missing_pct
    })
    
    display(result[result["Valeurs manquantes"] > 0])


### Train


,Valeurs manquantes,Pourcentage (%)
subject,2,0.02
speaker,2,0.02
job_title,2898,28.30
state,2210,21.58
party,2,0.02
barely_true_counts,2,0.02
false_counts,2,0.02
half_true_counts,2,0.02
mostly_true_counts,2,0.02
pants_on_fire_counts,2,0.02



### Valid


,Valeurs manquantes,Pourcentage (%)
job_title,345,26.87
state,279,21.73
context,12,0.93



### Test


,Valeurs manquantes,Pourcentage (%)
job_title,325,25.65
state,262,20.68
context,17,1.34


Après analyse des 3 datasets (train, valid, test), on identifie les valeurs manquantes suivantes

**`job_title` et `state` (≈25%) → on ne fait rien**
Ces colonnes ne seront pas utilisées dans notre pipeline de classification.
Le texte principal (`statement`) suffit pour entraîner le modèle.
Supprimer ces lignes ferait perdre 25% des données inutilement.

**`context`, `speaker`, `party`, `subject` (< 1%) → on remplace par une valeur neutre**
Ces colonnes sont utiles notamment pour l'analyse des biais (par parti, par speaker).
On remplace les manquants par `"unknown"` ou `""` pour ne perdre aucune ligne.
La règle d'or : on ne supprime une ligne que si elle est complètement inutilisable.

**`statement` et `label` → aucune action nécessaire**
Ces deux colonnes sont les plus importantes du projet :
`statement` est le texte qu'on classifie, `label` est ce qu'on cherche à prédire.
Heureusement elles ne contiennent aucune valeur manquante sur les 3 datasets.

Nettoyage des données

In [55]:
for df in [train, valid, test]:
    df["context"] = df["context"].fillna("")
    df["speaker"] = df["speaker"].fillna("unknown")
    df["party"]   = df["party"].fillna("unknown")
    df["subject"] = df["subject"].fillna("")

print(" Nettoyage terminé ")
print(f"   Train : {len(train)} lignes")
print(f"   Valid : {len(valid)} lignes")
print(f"   Test  : {len(test)} lignes")

# Vérification : plus aucune valeur manquante sur les colonnes utiles
cols_utiles = ["label", "statement", "speaker", "party", "context", "subject"]
print("\n### Vérification finale sur les colonnes utiles")
for nom, df in [("Train", train), ("Valid", valid), ("Test", test)]:
    nb = df[cols_utiles].isnull().sum().sum()
    print(f"   {nom} : {nb} valeurs manquantes restantes")

 Nettoyage terminé 
   Train : 10240 lignes
   Valid : 1284 lignes
   Test  : 1267 lignes

### Vérification finale sur les colonnes utiles
   Train : 0 valeurs manquantes restantes
   Valid : 0 valeurs manquantes restantes
   Test : 0 valeurs manquantes restantes


## Distribution des labels

On analyse ici la répartition des 6 labels de véracité dans le train set.
C'est une étape cruciale pour détecter un éventuel déséquilibre des classes
qui pourrait biaiser notre modèle.

In [56]:

import plotly.express as px

ORDRE_LABELS = [
    "pants-fire",
    "false",
    "barely-true",
    "half-true",
    "mostly-true",
    "true"
]

COULEURS = [
    "#d73027", 
    "#f46d43",  
    "#fdae61", 
    "#fee08b",  
    "#a6d96a", 
    "#1a9850"   
]

counts = train["label"].value_counts().reindex(ORDRE_LABELS)

fig = px.bar(
    x=counts.index,
    y=counts.values,
    color=counts.index,
    color_discrete_sequence=COULEURS,
    title="Distribution des labels — Train set",
    labels={"x": "Label de véracité", "y": "Nombre de déclarations"},
    text=counts.values
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

# Résumé chiffré
print("\n### Résumé")
for label, count in counts.items():
    pct = count / len(train) * 100
    print(f"   {label:<20} : {count} ({pct:.1f}%)")


### Résumé
   pants-fire           : 839 (8.2%)
   false                : 1995 (19.5%)
   barely-true          : 1654 (16.2%)
   half-true            : 2114 (20.6%)
   mostly-true          : 1962 (19.2%)
   true                 : 1676 (16.4%)


**Bonne nouvelle** : la majorité des classes sont relativement équilibrées
entre 16% et 20%, ce qui est favorable pour l'entraînement du modèle.

**Point d'attention** : `pants-fire` est sous-représenté avec seulement 8.2%.
C'est deux fois moins que les autres classes. Le modèle aura donc
moins d'exemples pour apprendre à détecter les mensonges flagrants.

On devra gérer ce léger déséquilibre avec l'une de ces stratégies :
- `class_weight="balanced"` dans scikit-learn
- Regroupement des labels en binaire (fake / true)
- Oversampling de la classe minoritaire


A voir après 

##  Analyse des speakers et des partis politiques

On analyse ici qui parle le plus dans le dataset et
quel est le lien entre le parti politique et la véracité des déclarations.
C'est aussi la base de notre analyse des biais plus tard.

In [57]:
top_speakers = train["speaker"].value_counts().head(10).reset_index()
top_speakers.columns = ["speaker", "count"]

fig = px.bar(
    top_speakers,
    x="count",
    y="speaker",
    orientation="h",
    title="Top 10 speakers les plus présents",
    labels={"count": "Nombre de déclarations", "speaker": "Speaker"},
    color="count",
    color_continuous_scale="Blues",
    text="count"
)

fig.update_traces(textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
fig.show()

Barack Obama est de loin le speaker le plus représenté avec 488 déclarations,
suivi de Donald Trump (273) et Hillary Clinton (239).
On note aussi "chain-email" qui représente des emails viraux anonymes.

Biais potentiel : le modèle risque de sur-apprendre
les patterns propres à Obama vu sa surreprésentation dans le dataset.

In [58]:

top_partis = train["party"].value_counts().head(8).reset_index()
top_partis.columns = ["party", "count"]

fig = px.bar(
    top_partis,
    x="party",
    y="count",
    title="Distribution des déclarations par parti politique",
    labels={"count": "Nombre de déclarations", "party": "Parti"},
    color="count",
    color_continuous_scale="Purples",
    text="count"
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

Le dataset est dominé par deux partis :
- Republican : 4497 déclarations (44%)
- Democrat : 3336 déclarations (33%)
- None : 1744 déclarations (17%)

Les autres partis (independent, libertarian, activist) sont très minoritaires.

Biais potentiel : le modèle sera beaucoup mieux calibré
pour les déclarations républicaines et démocrates que pour les autres.

In [59]:

# Garder uniquement les 5 partis les plus représentés
top_5_partis = train["party"].value_counts().head(5).index

df_partis = train[train["party"].isin(top_5_partis)]

# Calculer la proportion de chaque label par parti
df_grouped = (
    df_partis
    .groupby(["party", "label"])
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    df_grouped,
    x="party",
    y="count",
    color="label",
    title="Véracité des déclarations par parti politique (top 5)",
    labels={"count": "Nombre de déclarations", "party": "Parti", "label": "Label"},
    category_orders={"label": ORDRE_LABELS},
    color_discrete_sequence=COULEURS,
    barmode="group"
)

fig.show()

On observe une différence notable entre les partis :
- Les républicains ont plus de `pants-fire` et `false` que les démocrates
- Les démocrates ont proportionnellement plus de `mostly-true` 

Attention : cela ne veut pas dire qu'un parti ment plus que l'autre.
Cela peut refléter un biais dans les sources annotées par PolitiFact,
qui a peut-être vérifié davantage de déclarations républicaines controversées.

## Analyse de la longueur des déclarations

On analyse ici la longueur des déclarations en nombre de mots.
C'est important car les textes courts posent un défi particulier
pour les modèles NLP qui ont besoin de contexte pour bien classifier.

In [60]:

# Calcul du nombre de mots
train["nb_mots"] = train["statement"].str.split().str.len()

print("### Statistiques de longueur")
print(train["nb_mots"].describe().round(2))

# Histogramme
fig = px.histogram(
    train,
    x="nb_mots",
    nbins=50,
    title="Distribution de la longueur des déclarations",
    labels={"nb_mots": "Nombre de mots", "count": "Fréquence"},
    color_discrete_sequence=["#4C72B0"]
)

fig.add_vline(
    x=train["nb_mots"].mean(),
    line_dash="dash",
    line_color="red",
    annotation_text=f"Moyenne : {train['nb_mots'].mean():.1f} mots",
    annotation_position="top right"
)

fig.show()

### Statistiques de longueur
count    10240.00
mean        18.01
std          9.66
min          2.00
25%         12.00
50%         17.00
75%         22.00
max        467.00
Name: nb_mots, dtype: float64


es déclarations sont en moyenne très courtes : seulement 18 mots.
75% des déclarations font moins de 22 mots ce qui confirme
que le dataset est composé majoritairement de phrases courtes.

Défi pour la modélisation : les textes courts contiennent
peu de contexte, ce qui rend la classification plus difficile.
C'est une des raisons pour lesquelles BERT sera plus performant
que TF-IDF — il capte mieux le sens avec peu de mots.

Valeur maximale de 467 mots : il existe quelques outliers
très longs qui pourraient perturber certains modèles.
On les gardera mais on en tiendra compte dans l'analyse des erreurs.

In [61]:

moyenne_par_label = (
    train.groupby("label")["nb_mots"]
    .mean()
    .reindex(ORDRE_LABELS)
    .round(1)
    .reset_index()
)
moyenne_par_label.columns = ["label", "moyenne_mots"]

fig = px.bar(
    moyenne_par_label,
    x="label",
    y="moyenne_mots",
    color="label",
    title="Longueur moyenne des déclarations par label",
    labels={"moyenne_mots": "Nombre de mots moyen", "label": "Label"},
    color_discrete_sequence=COULEURS,
    text="moyenne_mots"
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

Les différences de longueur entre les labels sont très faibles
(entre 17 et 18.8 mots en moyenne).

On ne peut donc pas utiliser la longueur comme feature
discriminante pour détecter les fake news.

Cela confirme que le modèle devra s'appuyer uniquement
sur le contenu sémantique des mots et non sur la longueur.

## 🧹 Prétraitement du texte

Avant de vectoriser le texte, on doit le nettoyer.
Les modèles NLP sont sensibles aux majuscules, à la ponctuation
et aux mots très fréquents qui n'apportent pas de sens (stopwords).

### Étapes du prétraitement
1. Mise en minuscules
2. Suppression de la ponctuation
3. Suppression des stopwords
4. Tokenisation

In [62]:
import re
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))

def preprocess(text):
    # 1. Minuscules
    text = text.lower()
    # 2. Remplacer les tirets par un espace 
    text = text.replace("-", " ")
    # 3. Suppression de la ponctuation et chiffres
    text = re.sub(r"[^a-z\s]", "", text)
    # 4. Tokenisation
    tokens = text.split()
    # 5. Suppression des stopwords
    tokens = [t for t in tokens if t not in STOPWORDS]
    # 6. Rejoindre les tokens
    return " ".join(tokens)

# Appliquer sur les 3 datasets
train["statement_clean"] = train["statement"].apply(preprocess)
valid["statement_clean"] = valid["statement"].apply(preprocess)
test["statement_clean"]  = test["statement"].apply(preprocess)

print(" Prétraitement terminé !")
print("\n### Exemple avant / après")
print(f"Avant : {train['statement'].iloc[0]}")
print(f"Après : {train['statement_clean'].iloc[0]}")

 Prétraitement terminé !

### Exemple avant / après
Avant : Says the Annies List political group supports third-trimester abortions on demand.
Après : says annies list political group supports third trimester abortions demand


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gerau\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


##  Résultat du prétraitement

### Exemple
| | Texte |
|---|---|
| **Avant** | Says the Annies List political group supports third-trimester abortions on demand. |
| **Après** | says annies list political group supports third trimester abortions demand |

### Ce qui a été fait
-  Mise en minuscules
-  Tirets remplacés par des espaces
-  Ponctuation supprimée
-  Stopwords supprimés (the, on, a, is...)

### Ce qu'on garde volontairement
- Les noms propres (annies, list) → peuvent être des signaux importants
- Les mots courts non stopwords (demand, says) → porteurs de sens

## 🔢 Vectorisation TF-IDF + Premier modèle

On transforme maintenant le texte nettoyé en vecteurs numériques
avec TF-IDF puis on entraîne une Logistic Regression.
C'est notre baseline — le modèle le plus simple possible.

In [63]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. VECTORISATION TF-IDF
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # unigrammes + bigrammes
    max_features=50000,   # garder les 50 000 features les plus importantes
    min_df=2              # ignorer les mots qui apparaissent moins de 2 fois
)

X_train = tfidf.fit_transform(train["statement_clean"])
X_valid = tfidf.transform(valid["statement_clean"])
X_test  = tfidf.transform(test["statement_clean"])

y_train = train["label"]
y_valid = valid["label"]
y_test  = test["label"]

print(" Vectorisation TF-IDF terminée !")
print(f"   Taille X_train : {X_train.shape}")
print(f"   Taille X_valid : {X_valid.shape}")
print(f"   Taille X_test  : {X_test.shape}")



 Vectorisation TF-IDF terminée !
   Taille X_train : (10240, 15355)
   Taille X_valid : (1284, 15355)
   Taille X_test  : (1267, 15355)


## TF-IDF + Logistic Regression (6 classes)

In [64]:
# 2. ENTRAÎNEMENT
lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",  # gérer le déséquilibre des classes
    random_state=42
)

lr.fit(X_train, y_train)
print("\n Modèle entraîné !")

# 3. ÉVALUATION SUR VALID
y_pred_valid = lr.predict(X_valid)

print("\n### Résultats sur le Valid set")
print(f"Accuracy : {accuracy_score(y_valid, y_pred_valid):.4f}")
print("\nClassification Report :")
print(classification_report(y_valid, y_pred_valid, target_names=ORDRE_LABELS))


 Modèle entraîné !

### Résultats sur le Valid set
Accuracy : 0.2445

Classification Report :
              precision    recall  f1-score   support

  pants-fire       0.21      0.21      0.21       237
       false       0.27      0.24      0.25       263
 barely-true       0.24      0.19      0.21       248
   half-true       0.32      0.28      0.30       251
 mostly-true       0.22      0.32      0.26       116
        true       0.20      0.29      0.24       169

    accuracy                           0.24      1284
   macro avg       0.24      0.25      0.24      1284
weighted avg       0.25      0.24      0.24      1284



##  Résultats Baseline — TF-IDF + Logistic Regression (6 classes)

### Résultats obtenus

| Métrique | Valeur |
|---|---|
| Accuracy | 24.5% |
| F1-score macro | 0.24 |

### Pourquoi 24% c'est normal

Avec 6 classes équilibrées, un modèle aléatoire obtiendrait
1/6 = 16.7%. Notre modèle fait mieux que le hasard 

La tâche à 6 classes est intrinsèquement difficile car :
- Les frontières entre les labels sont floues
  (quelle différence entre "barely-true" et "half-true" ?)
- Les déclarations sont très courtes (18 mots en moyenne)
- TF-IDF ne capture pas le sens contextuel des mots

### Ce qu'on observe dans le classification report

- `half-true` est le mieux classifié (F1 = 0.30) car c'est
  la classe la plus représentée dans le train
- `pants-fire` et `barely-true` sont les plus difficiles
  car leurs frontières avec `false` sont ambiguës
- Le modèle a tendance à prédire `mostly-true` et `true`
  plus souvent (recall élevé) car ce sont des patterns
  plus fréquents dans le texte

### Prochaines étapes

Ce résultat est notre baseline. On va maintenant :
1. Tester la classification binaire (fake / true)
   → tâche plus simple, accuracy attendue ~60-65%
2. Tester d'autres modèles (Random Forest, XGBoost)
3. Tester Word2Vec / GloVe
4. Comparer tous les résultats

##  Classification Binaire — Fake vs True

On regroupe les 6 labels en 2 catégories :
- FAKE : pants-fire, false, barely-true
- TRUE : half-true, mostly-true, true

Cela simplifie la tâche et permet d'obtenir
des performances bien meilleures.

In [65]:

FAKE_LABELS = ["pants-fire", "false", "barely-true"]
TRUE_LABELS = ["half-true", "mostly-true", "true"]

def to_binary(label):
    if label in FAKE_LABELS:
        return "fake"
    else:
        return "true"

train["label_binary"] = train["label"].apply(to_binary)
valid["label_binary"] = valid["label"].apply(to_binary)
test["label_binary"]  = test["label"].apply(to_binary)

print(" Labels binaires créés !")
print("\n### Distribution binaire — Train")
print(train["label_binary"].value_counts())
print("\n### Distribution binaire — Valid")
print(valid["label_binary"].value_counts())

 Labels binaires créés !

### Distribution binaire — Train
label_binary
true    5752
fake    4488
Name: count, dtype: int64

### Distribution binaire — Valid
label_binary
true    668
fake    616
Name: count, dtype: int64


In [66]:


y_train_bin = train["label_binary"]
y_valid_bin = valid["label_binary"]
y_test_bin  = test["label_binary"]

# Réutiliser le même TF-IDF déjà fitté
lr_bin = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_bin.fit(X_train, y_train_bin)
print("Modèle binaire entraîné !")

y_pred_bin = lr_bin.predict(X_valid)

print("\n### Résultats sur le Valid set")
print(f"Accuracy : {accuracy_score(y_valid_bin, y_pred_bin):.4f}")
print("\nClassification Report :")
print(classification_report(y_valid_bin, y_pred_bin, target_names=["fake", "true"]))

Modèle binaire entraîné !

### Résultats sur le Valid set
Accuracy : 0.6137

Classification Report :
              precision    recall  f1-score   support

        fake       0.59      0.61      0.60       616
        true       0.63      0.62      0.62       668

    accuracy                           0.61      1284
   macro avg       0.61      0.61      0.61      1284
weighted avg       0.61      0.61      0.61      1284



In [67]:


cm_bin = confusion_matrix(y_valid_bin, y_pred_bin, labels=["fake", "true"])

fig = ff.create_annotated_heatmap(
    z=cm_bin,
    x=["fake", "true"],
    y=["fake", "true"],
    colorscale="Blues",
    showscale=True
)

fig.update_layout(
    title="Matrice de confusion — Logistic Regression Binaire",
    xaxis_title="Prédit",
    yaxis_title="Réel",
    yaxis={"autorange": "reversed"}
)

fig.show()

##  Résultats Classification Binaire/6classes — TF-IDF + Logistic Regression

### Résultats obtenus

| Métrique | 6 classes | Binaire |
|---|---|---|
| Accuracy | 24.5% | 61.4% |
| F1-score macro | 0.24 | 0.61 |

### Interprétation

La classification binaire est bien meilleure que la classification
à 6 classes comme attendu. Avec 2 classes équilibrées,
un modèle aléatoire obtiendrait 50%, on est donc
11 points au dessus du hasard 

### Ce qu'on observe

- `fake` et `true` sont classifiés de manière équilibrée
  (precision et recall similaires pour les deux classes)
- Le modèle n'est pas biaisé vers une classe en particulier
- Les résultats sont cohérents entre precision et recall

### Limites de ce modèle baseline

- TF-IDF ne capture pas le sens contextuel des mots
- 61% reste modeste, on peut faire mieux avec :
  - Random Forest ou XGBoost
  - Word2Vec / GloVe
  - BERT

## 🌲 Random Forest — 6 classes et Binaire

On teste Random Forest avec le même TF-IDF
pour comparer directement avec la Logistic Regression.

In [68]:

from sklearn.ensemble import RandomForestClassifier

# 1. RANDOM FOREST 6 CLASSES
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1  # utilise tous les coeurs CPU
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_valid)

print("### Random Forest — 6 classes")
print(f"Accuracy : {accuracy_score(y_valid, y_pred_rf):.4f}")
print(classification_report(y_valid, y_pred_rf, target_names=ORDRE_LABELS))

# 2. RANDOM FOREST BINAIRE
rf_bin = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_bin.fit(X_train, y_train_bin)
y_pred_rf_bin = rf_bin.predict(X_valid)

print("\n### Random Forest — Binaire")
print(f"Accuracy : {accuracy_score(y_valid_bin, y_pred_rf_bin):.4f}")
print(classification_report(y_valid_bin, y_pred_rf_bin, target_names=["fake", "true"]))

### Random Forest — 6 classes
Accuracy : 0.2508
              precision    recall  f1-score   support

  pants-fire       0.26      0.16      0.20       237
       false       0.25      0.38      0.30       263
 barely-true       0.22      0.24      0.23       248
   half-true       0.28      0.30      0.29       251
 mostly-true       0.33      0.14      0.19       116
        true       0.22      0.20      0.21       169

    accuracy                           0.25      1284
   macro avg       0.26      0.24      0.24      1284
weighted avg       0.26      0.25      0.24      1284


### Random Forest — Binaire
Accuracy : 0.6153
              precision    recall  f1-score   support

        fake       0.61      0.53      0.57       616
        true       0.62      0.69      0.65       668

    accuracy                           0.62      1284
   macro avg       0.62      0.61      0.61      1284
weighted avg       0.62      0.62      0.61      1284



In [69]:

resultats = {
    "Modèle"   : ["LR + TF-IDF", "LR + TF-IDF", "RF + TF-IDF", "RF + TF-IDF"],
    "Classes"  : ["6 classes", "Binaire", "6 classes", "Binaire"],
    "Accuracy" : [
        accuracy_score(y_valid, y_pred_valid),
        accuracy_score(y_valid_bin, y_pred_bin),
        accuracy_score(y_valid, y_pred_rf),
        accuracy_score(y_valid_bin, y_pred_rf_bin)
    ]
}

df_resultats = pd.DataFrame(resultats)
df_resultats["Accuracy"] = df_resultats["Accuracy"].round(4)

print(df_resultats)


        Modèle    Classes  Accuracy
0  LR + TF-IDF  6 classes    0.2445
1  LR + TF-IDF    Binaire    0.6137
2  RF + TF-IDF  6 classes    0.2508
3  RF + TF-IDF    Binaire    0.6153


##  Comparaison LR vs Random Forest — TF-IDF

### Résultats obtenus

| Modèle | 6 classes | Binaire |
|---|---|---|
| LR + TF-IDF | 24.45% | 61.37% |
| RF + TF-IDF | 25.08% | 61.53% |

### Interprétation

Les deux modèles donnent des résultats très proches !

- En 6 classes : RF légèrement meilleur (+0.6%)
- En binaire : RF légèrement meilleur (+0.2%)

### Pourquoi Random Forest ne fait pas mieux ?

TF-IDF produit des vecteurs très creux (sparse) avec
15 355 features. Random Forest n'est pas le modèle
le plus adapté à ce type de données car il a du mal
à exploiter efficacement des espaces de très haute dimension.

La Logistic Regression est souvent plus performante
que Random Forest sur des données TF-IDF.

### Prochaine étape

On teste XGBoost qui gère mieux les espaces de haute
dimension

In [70]:

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Encoder les labels en chiffres (XGBoost ne prend pas de strings)
le_6 = LabelEncoder()
le_bin = LabelEncoder()

y_train_enc     = le_6.fit_transform(y_train)
y_valid_enc     = le_6.transform(y_valid)

y_train_bin_enc = le_bin.fit_transform(y_train_bin)
y_valid_bin_enc = le_bin.transform(y_valid_bin)

# 1. XGBOOST 6 CLASSES
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train_enc)
y_pred_xgb_enc = xgb.predict(X_valid)
y_pred_xgb = le_6.inverse_transform(y_pred_xgb_enc)

print("### XGBoost — 6 classes")
print(f"Accuracy : {accuracy_score(y_valid, y_pred_xgb):.4f}")
print(classification_report(y_valid, y_pred_xgb, target_names=ORDRE_LABELS))

# 2. XGBOOST BINAIRE
xgb_bin = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_bin.fit(X_train, y_train_bin_enc)
y_pred_xgb_bin_enc = xgb_bin.predict(X_valid)
y_pred_xgb_bin = le_bin.inverse_transform(y_pred_xgb_bin_enc)

print("\n### XGBoost — Binaire")
print(f"Accuracy : {accuracy_score(y_valid_bin, y_pred_xgb_bin):.4f}")
print(classification_report(y_valid_bin, y_pred_xgb_bin, target_names=["fake", "true"]))

c:\Users\gerau\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning:

[15:43:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.




### XGBoost — 6 classes
Accuracy : 0.2609
              precision    recall  f1-score   support

  pants-fire       0.18      0.08      0.11       237
       false       0.27      0.47      0.34       263
 barely-true       0.27      0.32      0.29       248
   half-true       0.27      0.29      0.28       251
 mostly-true       0.33      0.09      0.14       116
        true       0.23      0.18      0.20       169

    accuracy                           0.26      1284
   macro avg       0.26      0.24      0.23      1284
weighted avg       0.25      0.26      0.24      1284


### XGBoost — Binaire
Accuracy : 0.5872
              precision    recall  f1-score   support

        fake       0.66      0.29      0.40       616
        true       0.57      0.86      0.69       668

    accuracy                           0.59      1284
   macro avg       0.61      0.58      0.54      1284
weighted avg       0.61      0.59      0.55      1284



In [71]:

resultats = {
    "Modèle"   : [
        "LR + TF-IDF", "LR + TF-IDF",
        "RF + TF-IDF", "RF + TF-IDF",
        "XGB + TF-IDF", "XGB + TF-IDF"
    ],
    "Classes"  : ["6 classes", "Binaire"] * 3,
    "Accuracy" : [
        accuracy_score(y_valid, y_pred_valid),
        accuracy_score(y_valid_bin, y_pred_bin),
        accuracy_score(y_valid, y_pred_rf),
        accuracy_score(y_valid_bin, y_pred_rf_bin),
        accuracy_score(y_valid, y_pred_xgb),
        accuracy_score(y_valid_bin, y_pred_xgb_bin)
    ]
}

df_resultats = pd.DataFrame(resultats)
df_resultats["Accuracy"] = df_resultats["Accuracy"].round(4)

print(df_resultats)

fig = px.bar(
    df_resultats,
    x="Modèle",
    y="Accuracy",
    color="Classes",
    barmode="group",
    title="Comparaison des 3 modèles — TF-IDF — Valid set",
    text="Accuracy",
    color_discrete_sequence=["#4C72B0", "#DD8452"]
)

fig.update_traces(textposition="outside")
fig.update_layout(yaxis_range=[0, 1])
fig.show()

         Modèle    Classes  Accuracy
0   LR + TF-IDF  6 classes    0.2445
1   LR + TF-IDF    Binaire    0.6137
2   RF + TF-IDF  6 classes    0.2508
3   RF + TF-IDF    Binaire    0.6153
4  XGB + TF-IDF  6 classes    0.2609
5  XGB + TF-IDF    Binaire    0.5872


## Choix de la structure de classification

Après comparaison des deux approches, on choisit
de se concentrer uniquement sur la classification binaire :

- FAKE : pants-fire, false, barely-true
- TRUE : half-true, mostly-true, true

### Pourquoi ce choix ?

1. La tâche à 6 classes est trop ambiguë — les frontières
   entre barely-true, half-true et mostly-true sont floues
   même pour un humain

2. Les performances à 6 classes plafonnent à ~26%
   ce qui est trop faible pour être exploitable

3. La classification binaire est plus réaliste dans
   un contexte réel de détection de fake news

4. Les performances binaires (~61%) sont bien plus
   interprétables et comparables avec la littérature

### Récap des meilleurs résultats TF-IDF

| Modèle | Accuracy Binaire |
|---|---|
| LR + TF-IDF | 61.37% |
| RF + TF-IDF | 61.53% ✅ meilleur |
| XGB + TF-IDF | 58.72% |

## Vectorisation Word2Vec

On remplace TF-IDF par Word2Vec qui capture
le sens sémantique des mots contrairement à TF-IDF.

On utilise un modèle Word2Vec pré-entraîné sur
Google News (3 millions de mots, 300 dimensions).

In [72]:
import gensim.downloader as api
import numpy as np

print("Téléchargement du modèle Word2Vec...")
print("Cela peut prendre quelques minutes la première fois)")

w2v_model = api.load("word2vec-google-news-300")

print(" Modèle Word2Vec chargé !")
print(f"   Vocabulaire : {len(w2v_model)} mots")
print(f"   Dimensions  : {w2v_model.vector_size}")

Téléchargement du modèle Word2Vec...
Cela peut prendre quelques minutes la première fois)
 Modèle Word2Vec chargé !
   Vocabulaire : 3000000 mots
   Dimensions  : 300


In [73]:

def vectorize_w2v(text, model):
    tokens = text.split()
    # Garder uniquement les mots présents dans le modèle
    vectors = [model[word] for word in tokens if word in model]
    if len(vectors) == 0:
        # Si aucun mot trouvé retourner un vecteur de zéros
        return np.zeros(model.vector_size)
    # Moyenne des vecteurs de tous les mots
    return np.mean(vectors, axis=0)

print("Vectorisation en cours...")

X_train_w2v = np.array([vectorize_w2v(text, w2v_model) for text in train["statement_clean"]])
X_valid_w2v = np.array([vectorize_w2v(text, w2v_model) for text in valid["statement_clean"]])
X_test_w2v  = np.array([vectorize_w2v(text, w2v_model) for text in test["statement_clean"]])

print("Vectorisation Word2Vec terminée !")
print(f"Taille X_train : {X_train_w2v.shape}")
print(f"Taille X_valid : {X_valid_w2v.shape}")
print(f"Taille X_test  : {X_test_w2v.shape}")

Vectorisation en cours...
Vectorisation Word2Vec terminée !
Taille X_train : (10240, 300)
Taille X_valid : (1284, 300)
Taille X_test  : (1267, 300)


In [74]:

# 1. LOGISTIC REGRESSION
lr_w2v = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
lr_w2v.fit(X_train_w2v, y_train_bin)
y_pred_lr_w2v = lr_w2v.predict(X_valid_w2v)

print("### LR + Word2Vec — Binaire")
print(f"Accuracy : {accuracy_score(y_valid_bin, y_pred_lr_w2v):.4f}")

# 2. RANDOM FOREST
rf_w2v = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_w2v.fit(X_train_w2v, y_train_bin)
y_pred_rf_w2v = rf_w2v.predict(X_valid_w2v)

print("\n### RF + Word2Vec — Binaire")
print(f"Accuracy : {accuracy_score(y_valid_bin, y_pred_rf_w2v):.4f}")

# 3. XGBOOST
xgb_w2v = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb_w2v.fit(X_train_w2v, y_train_bin_enc)
y_pred_xgb_w2v_enc = xgb_w2v.predict(X_valid_w2v)
y_pred_xgb_w2v = le_bin.inverse_transform(y_pred_xgb_w2v_enc)

print("\n### XGB + Word2Vec — Binaire")
print(f"Accuracy : {accuracy_score(y_valid_bin, y_pred_xgb_w2v):.4f}")

### LR + Word2Vec — Binaire
Accuracy : 0.6005

### RF + Word2Vec — Binaire
Accuracy : 0.5872

### XGB + Word2Vec — Binaire
Accuracy : 0.5989


##  Comparaison TF-IDF vs Word2Vec — Résultats

### Résultats obtenus

| Modèle | TF-IDF | Word2Vec |
|---|---|---|
| Logistic Regression | 61.37% | 60.05% |
| Random Forest | 61.53% (LE BEST) | 58.72% |
| XGBoost | 58.72% | 59.89% |

### Interprétation

Surprise : TF-IDF fait globalement mieux que Word2Vec !

Cela s'explique par plusieurs raisons :

1. Les déclarations sont très courtes (18 mots en moyenne)
   Word2Vec fait la moyenne des vecteurs de tous les mots
   ce qui perd beaucoup d'information sur des textes courts

2. TF-IDF capture les mots rares mais discriminants
   comme des noms propres politiques spécifiques
   que Word2Vec noie dans la moyenne

3. Word2Vec Google News est entraîné sur des articles
   longs, pas sur des déclarations politiques courtes

### Warning XGBoost

Le warning use_label_encoder est sans gravité,
c'est juste un paramètre déprécié dans les nouvelles
versions de XGBoost. On peut le retirer du code.

### Meilleur modèle jusqu'ici

Random Forest + TF-IDF → 61.53% 

### Prochaine étape


In [81]:
# ============================================================
# CELLULE 28 - VÉRIFICATION GPU
# ============================================================

import torch

print(f"PyTorch version : {torch.__version__}")
print(f"GPU disponible  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Nom du GPU      : {torch.cuda.get_device_name(0)}")
    print(f"Mémoire GPU     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = torch.device("cuda")
else:
    print("⚠️ Pas de GPU détecté, on utilise le CPU")
    device = torch.device("cpu")

print(f"\n✅ Device utilisé : {device}")

PyTorch version : 2.10.0+cpu
GPU disponible  : False
⚠️ Pas de GPU détecté, on utilise le CPU

✅ Device utilisé : cpu
